# Diffusion Policy Push-T Training on Colab (Custom Implementation)

In [1]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Setup checkpoint directory on Drive
import os
CKPT_DIR = '/content/drive/MyDrive/diffusion_policy_checkpoints'
os.makedirs(CKPT_DIR, exist_ok=True)
print(f'Checkpoint dir: {CKPT_DIR}')

Mounted at /content/drive
Checkpoint dir: /content/drive/MyDrive/diffusion_policy_checkpoints


In [2]:
# Check GPU
import torch
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

CUDA: True
GPU: Tesla T4
VRAM: 15.6 GB


In [3]:
# Clone custom implementation from GitHub
!git clone https://github.com/VyDat-1702/Diffusion-Policy.git /content/diffusion
%cd /content/diffusion
!ls -la

Cloning into '/content/diffusion'...
remote: Enumerating objects: 954, done.
remote: Counting objects: 100% (954/954), done.
remote: Compressing objects: 100% (900/900), done.
remote: Total 954 (delta 57), reused 938 (delta 41), pack-reused 0 (from 0)
Receiving objects: 100% (954/954), 30.62 MiB | 19.43 MiB/s, done.
Resolving deltas: 100% (57/57), done.
/content/diffusion
total 184
drwxr-xr-x 10 root root  4096 Aug 30 00:27 .
drwxr-xr-x  1 root root  4096 Aug 30 00:26 ..
drwxr-xr-x  2 root root  4096 Aug 30 00:27 checkpoints
drwxr-xr-x  2 root root  4096 Aug 30 00:27 common
drwxr-xr-x  3 root root  4096 Aug 30 00:27 data
-rw-r--r--  1 root root 77876 Aug 30 00:27 diffusion_colab.ipynb
drwxr-xr-x  2 root root  4096 Aug 30 00:27 envs
-rw-r--r--  1 root root  4684 Aug 30 00:27 evaluate.py
drwxr-xr-x  8 root root  4096 Aug 30 00:27 .git
-rw-r--r--  1 root root   225 Aug 30 00:27 .gitignore
-rw-r--r--  1 root root  6911 Aug 30 00:27 infer_denoise.py
drwxr-xr-x  2 root root  4096 Aug 30 00:2

In [4]:
# Install dependencies
!pip install einops diffusers zarr pygame-ce pymunk -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.7/363.7 kB 32.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.1/12.1 MB 121.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 62.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 135.7 MB/s eta 0:00:00


In [5]:
# Download Push-T dataset
!mkdir -p data/pusht
!wget -q https://diffusion-policy.cs.columbia.edu/data/training/pusht.zip -O /tmp/pusht.zip
!unzip -q /tmp/pusht.zip -d data/pusht/
!ls -la data/pusht/

total 16
drwxr-xr-x 4 root root 4096 Aug 30 00:27 .
drwxr-xr-x 3 root root 4096 Aug 30 00:27 ..
drwx------ 3 root root 4096 Feb 27  2023 pusht
drwxr-xr-x 4 root root 4096 Aug 30 00:27 pusht_cchi_v7_replay.zarr


In [6]:
# Train custom implementation (500 epochs)
# Checkpoints saved to Google Drive automatically
%cd /content/diffusion
!python train_ddpm.py \
    --epochs 500 \
    --batch_size 256 \
    --device cuda \
    --use_ema \
    --lr_warmup_steps 500 \
    --zarr_path data/pusht/pusht_cchi_v7_replay.zarr \
    --ckpt_dir /content/drive/MyDrive/diffusion_policy_checkpoints

/content/diffusion
Model: U-Net 1D | Params: 65,353,218
Epoch 1/500: 100% 80/80 [00:23<00:00,  3.46it/s]
Epoch 1/500 | Train Loss: 0.902873 | LR: 1.60e-05
Epoch 1/500 | Val Loss: 0.722678 | LR: 1.60e-05
Saved best checkpoint to /content/drive/MyDrive/diffusion_policy_checkpoints/diffusion_policy_best.pt
Epoch 2/500: 100% 80/80 [00:23<00:00,  3.46it/s]
Epoch 2/500 | Train Loss: 0.203443 | LR: 3.20e-05
Epoch 2/500 | Val Loss: 0.142894 | LR: 3.20e-05
Saved best checkpoint to /content/drive/MyDrive/diffusion_policy_checkpoints/diffusion_policy_best.pt
Epoch 3/500: 100% 80/80 [00:24<00:00,  3.26it/s]
Epoch 3/500 | Train Loss: 0.083901 | LR: 4.80e-05
Epoch 3/500 | Val Loss: 0.085406 | LR: 4.80e-05
Saved best checkpoint to /content/drive/MyDrive/diffusion_policy_checkpoints/diffusion_policy_best.pt
Epoch 4/500: 100% 80/80 [00:23<00:00,  3.37it/s]
Epoch 4/500 | Train Loss: 0.068711 | LR: 6.40e-05
Epoch 4/500 | Val Loss: 0.062481 | LR: 6.40e-05
Saved best checkpoint to /content/drive/MyDrive/di

In [7]:
# Evaluate after training (load from Google Drive)
%cd /content/diffusion
!python evaluate.py \
    --num_episodes 50 \
    --device cuda \
    --ckpt_path /content/drive/MyDrive/diffusion_policy_checkpoints

/content/diffusion
pygame-ce 2.5.8 (SDL 2.32.10, Python 3.13.15)
Evaluating: 100% 50/50 [22:13<00:00, 26.68s/it]

=== Evaluation Results ===
Success Rate (coverage > 0.95): 54.00%
Avg Max Coverage Score: 0.8991
Avg Episode Length: 236.1
Avg Position Error: 21.95
Avg Angle Error: 0.1592


In [ ]:
# Visualize results (load from Google Drive)
%cd /content/diffusion
!python visualize.py --device cuda --ckpt_path /content/drive/MyDrive/diffusion_policy_checkpoints